In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print('device:', device)
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True


device: cuda


In [2]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data" / "processed_data"

train_df = pd.read_parquet(DATA_DIR / "train_processed.parquet")
valid_df = pd.read_parquet(DATA_DIR / "valid_processed.parquet")

print(train_df.shape, valid_df.shape)


(1614612, 140) (403740, 140)


In [3]:
TARGET_COL = 'target'
SPLIT_DROP_COLS = ['target', 'datetime', 'date', 'is_consumption', 'row_id']
ALL_DROP_COLS = ['target', 'datetime', 'date', 'row_id']

FEATURE_COLS_SPLIT = [c for c in train_df.columns if c not in SPLIT_DROP_COLS]
FEATURE_COLS_ALL = [c for c in train_df.columns if c not in ALL_DROP_COLS]
print('split features:', len(FEATURE_COLS_SPLIT))
print('global features:', len(FEATURE_COLS_ALL))


split features: 135
global features: 136


In [4]:
train_df = train_df.sort_values(['prediction_unit_id', 'is_consumption', 'datetime']).reset_index(drop=True)
valid_df = valid_df.sort_values(['prediction_unit_id', 'is_consumption', 'datetime']).reset_index(drop=True)

train_cons = train_df[train_df.is_consumption == 1].copy()
valid_cons = valid_df[valid_df.is_consumption == 1].copy()
train_prod = train_df[train_df.is_consumption == 0].copy()
valid_prod = valid_df[valid_df.is_consumption == 0].copy()
train_all = train_df.copy()
valid_all = valid_df.copy()

print('train_cons:', train_cons.shape, 'train_prod:', train_prod.shape, 'train_all:', train_all.shape)
print('valid_cons:', valid_cons.shape, 'valid_prod:', valid_prod.shape, 'valid_all:', valid_all.shape)

train_cons: (807306, 140) train_prod: (807306, 140) train_all: (1614612, 140)
valid_cons: (201870, 140) valid_prod: (201870, 140) valid_all: (403740, 140)


In [5]:
scaler_cons = StandardScaler()
scaler_prod = StandardScaler()
scaler_all = StandardScaler()

train_cons[FEATURE_COLS_SPLIT] = scaler_cons.fit_transform(train_cons[FEATURE_COLS_SPLIT])
valid_cons[FEATURE_COLS_SPLIT] = scaler_cons.transform(valid_cons[FEATURE_COLS_SPLIT])

train_prod[FEATURE_COLS_SPLIT] = scaler_prod.fit_transform(train_prod[FEATURE_COLS_SPLIT])
valid_prod[FEATURE_COLS_SPLIT] = scaler_prod.transform(valid_prod[FEATURE_COLS_SPLIT])

train_all[FEATURE_COLS_ALL] = scaler_all.fit_transform(train_all[FEATURE_COLS_ALL])
valid_all[FEATURE_COLS_ALL] = scaler_all.transform(valid_all[FEATURE_COLS_ALL])


In [6]:
SEQ_LEN = 24

class SequenceDataset(Dataset):
    def __init__(self, df, feature_cols, seq_len=24, group_cols=None):
        self.seq_len = seq_len
        self.group_cols = group_cols or ['prediction_unit_id', 'is_consumption']
        self.groups = []
        self.sample_map = []

        for _, g in df.groupby(self.group_cols, sort=False):
            g = g.sort_values('datetime').reset_index(drop=True)
            feat = g[feature_cols].to_numpy(dtype=np.float32)
            targ = g[TARGET_COL].to_numpy(dtype=np.float32)
            row_ids = (g['row_id'].to_numpy() if 'row_id' in g.columns else g.index.to_numpy()).astype(np.int64)
            group_idx = len(self.groups)
            self.groups.append({'X': feat, 'y': targ, 'row_id': row_ids})
            for end_idx in range(seq_len, len(g)):
                self.sample_map.append((group_idx, end_idx))

    def __len__(self):
        return len(self.sample_map)

    def __getitem__(self, idx):
        group_idx, end_idx = self.sample_map[idx]
        group = self.groups[group_idx]
        x = group['X'][end_idx - self.seq_len:end_idx]
        y = group['y'][end_idx]
        row_id = group['row_id'][end_idx]
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(row_id, dtype=torch.long),
        )

In [7]:
train_cons_dataset = SequenceDataset(train_cons, FEATURE_COLS_SPLIT, SEQ_LEN)
valid_cons_dataset = SequenceDataset(valid_cons, FEATURE_COLS_SPLIT, SEQ_LEN)
train_prod_dataset = SequenceDataset(train_prod, FEATURE_COLS_SPLIT, SEQ_LEN)
valid_prod_dataset = SequenceDataset(valid_prod, FEATURE_COLS_SPLIT, SEQ_LEN)
train_all_dataset = SequenceDataset(train_all, FEATURE_COLS_ALL, SEQ_LEN)
valid_all_dataset = SequenceDataset(valid_all, FEATURE_COLS_ALL, SEQ_LEN)

print('train datasets:', len(train_cons_dataset), len(train_prod_dataset), len(train_all_dataset))
print('valid datasets:', len(valid_cons_dataset), len(valid_prod_dataset), len(valid_all_dataset))

train datasets: 805650 805650 1611300
valid datasets: 200214 200214 400428


In [8]:
BATCH_SIZE = 256
PIN_MEMORY = device.type == 'cuda'

def make_loader(dataset, shuffle):
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, pin_memory=PIN_MEMORY)

train_cons_loader = make_loader(train_cons_dataset, True)
valid_cons_loader = make_loader(valid_cons_dataset, False)
train_prod_loader = make_loader(train_prod_dataset, True)
valid_prod_loader = make_loader(valid_prod_dataset, False)
train_all_loader = make_loader(train_all_dataset, True)
valid_all_loader = make_loader(valid_all_dataset, False)

In [9]:
class GRURegressor(nn.Module):
    def __init__(self,input_dim,hidden_dim=64):
        super().__init__()
        self.gru=nn.GRU(input_dim,hidden_dim,num_layers=2,batch_first=True,dropout=0.2)
        self.fc=nn.Sequential(
            nn.Linear(hidden_dim,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )

    def forward(self,x):
        out,_=self.gru(x)
        out=out[:,-1,:]
        return self.fc(out).squeeze(1)


In [10]:
def make_model(input_dim):
    return GRURegressor(input_dim).to(device)

criterion = nn.L1Loss()
EPOCHS = 8


In [11]:
def train_epoch(model, loader, optimizer):
    model.train()
    total = 0.0
    for X, y, _ in loader:
        X = X.to(device, non_blocking=PIN_MEMORY)
        y = y.to(device, non_blocking=PIN_MEMORY)
        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total += loss.item() * X.size(0)
    return total / len(loader.dataset)

def predict(model, loader, pred_col):
    model.eval()
    preds, targets, row_ids = [], [], []
    with torch.no_grad():
        for X, y, row_id in loader:
            X = X.to(device, non_blocking=PIN_MEMORY)
            p = model(X).detach().cpu().numpy()
            preds.append(p)
            targets.append(y.numpy())
            row_ids.append(row_id.numpy())
    out = pd.DataFrame({
        'row_id': np.concatenate(row_ids),
        TARGET_COL: np.concatenate(targets),
        pred_col: np.concatenate(preds),
    })
    return out.sort_values('row_id').reset_index(drop=True)

def fit_segment(name, train_loader, valid_loader, input_dim, pred_col):
    model = make_model(input_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
    best_mae = np.inf
    best_state = None

    for epoch in range(EPOCHS):
        train_loss = train_epoch(model, train_loader, optimizer)
        pred_df = predict(model, valid_loader, pred_col)
        val_mae = mean_absolute_error(pred_df[TARGET_COL], pred_df[pred_col])
        print(f'[{name}] epoch={epoch + 1} train_loss={train_loss:.4f} val_mae={val_mae:.4f}')
        if val_mae < best_mae:
            best_mae = val_mae
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    pred_df = predict(model, valid_loader, pred_col)
    mae = mean_absolute_error(pred_df[TARGET_COL], pred_df[pred_col])
    return {'segment': name, 'model': model, 'pred_df': pred_df, 'mae': mae}

In [12]:
res_cons = fit_segment('consumption', train_cons_loader, valid_cons_loader, len(FEATURE_COLS_SPLIT), 'pred_gru')
res_prod = fit_segment('production', train_prod_loader, valid_prod_loader, len(FEATURE_COLS_SPLIT), 'pred_gru')
res_all = fit_segment('overall', train_all_loader, valid_all_loader, len(FEATURE_COLS_ALL), 'pred_gru')

[consumption] epoch=1 train_loss=212.9618 val_mae=142.8511
[consumption] epoch=2 train_loss=68.1393 val_mae=55.7338
[consumption] epoch=3 train_loss=37.0490 val_mae=42.1222
[consumption] epoch=4 train_loss=31.0089 val_mae=39.3841
[consumption] epoch=5 train_loss=28.7375 val_mae=36.7578
[consumption] epoch=6 train_loss=27.3991 val_mae=34.0729
[consumption] epoch=7 train_loss=26.5645 val_mae=33.9618
[consumption] epoch=8 train_loss=25.9963 val_mae=33.3807
[production] epoch=1 train_loss=31.2971 val_mae=57.6063
[production] epoch=2 train_loss=16.3085 val_mae=44.0485
[production] epoch=3 train_loss=13.5673 val_mae=37.2567
[production] epoch=4 train_loss=12.4895 val_mae=35.4446
[production] epoch=5 train_loss=11.9899 val_mae=33.4490
[production] epoch=6 train_loss=11.6890 val_mae=34.5239
[production] epoch=7 train_loss=11.4918 val_mae=33.2351
[production] epoch=8 train_loss=11.3338 val_mae=33.5973
[overall] epoch=1 train_loss=88.1089 val_mae=52.8792
[overall] epoch=2 train_loss=26.5151 val_

In [13]:
pred_split = pd.concat([res_prod['pred_df'], res_cons['pred_df']], axis=0).sort_values('row_id').reset_index(drop=True)
split_overall_mae = mean_absolute_error(pred_split[TARGET_COL], pred_split['pred_gru'])
pred_single = res_all['pred_df'].sort_values('row_id').set_index('row_id').loc[pred_split['row_id']].reset_index()
pred_split['pred_split_ensemble'] = 0.5 * pred_split['pred_gru'].values + 0.5 * pred_single['pred_gru'].values
ensemble_mae = mean_absolute_error(pred_split[TARGET_COL], pred_split['pred_split_ensemble'])

mae_summary = pd.DataFrame(
    {
        'MAE': [
            res_prod['mae'],
            res_cons['mae'],
            split_overall_mae,
            res_all['mae'],
            ensemble_mae,
        ]
    },
    index=['production', 'consumption', 'overall(split)', 'overall(single)', 'split-ensemble'],
)
display(mae_summary.style.format('{:.4f}'))

,MAE
production,33.2351
consumption,33.3807
overall(split),33.3079
overall(single),29.7967
split-ensemble,29.5482
